# Predicting Evals Y from Pile X

In [ ]:
import numpy as np
import os
import pickle
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm
from itertools import product
from sklearn.compose import TransformedTargetRegressor
import matplotlib.pyplot as plt
from collections import namedtuple

## Data

We have collected data on $n=58$ models from huggingface for which we have both:

* Token losses on different pile contexts (1300 contexts)
* Benchmark evaluations (~100 aggregated scores)

We can optionally drop some strangely behaving models and benchmarks here, leaving us with:

X: 56 x 1300 feature matrix
Y: 56 x 91 target matrix

The question is - to what extent does X predict Y? If "pile is all you need", the information in X should explain much of the information in Y.

When modeling signal in the data, the variance of the columns equates to importance - we take the common approach of scaling the columns such that they are "equally important", the way you might scale exam scores.

In [ ]:
# Load the data (preprocessed)
f = os.path.expanduser("~/code/lsoc/data/lsoc1-aligned.pkl")

with open(f, 'rb') as f:
   dat = pickle.load(f)

evals = dat['evals']
pile = dat['pile']

# Optional - drop outlier models
drop = ["google/gemma-7b-it", "google/gemma-2b-it"]
evals.drop(index=drop, inplace=True)
pile.drop(index=drop, inplace=True)

# Optional - drop outlier tasks where many small models fail completely
drop_cols = [c for c in evals.columns if "generative" in c or "cot" in c]
evals.drop(columns=drop_cols, inplace=True)

# Note on dropping data: doing so makes the problem "easier" so we might expect better reconstruction scores.

# Scaling encodes the relative "importance" of different columns
# If we take the position of "equal importance", we can just center and whiten the data
Xs = StandardScaler().fit_transform(pile.values)
Ys = StandardScaler().fit_transform(evals.values)

# These matrices aren't anywhere near full rank, because n << features
# PCA is one way of getting full rank forms
# (but should be re-PCA'd inside the pipeline to avoid label leakage)
Xc = PCA().fit_transform(Xs)  
Yc = PCA().fit_transform(Ys)

## Cross validation pipeline

For the remainder of the notebook we'll be looking at models to reconstruct Y (eval data) from X (pile data).

We do this with held-out validation because most error metrics reward a perfect fit.
With d=1300 features and n=56 samples, we have any number of models that memorise the data, but they probably aren't "good" models in terms of generalisation. The real test is whether a model works on unseen data.

Here we set up a *cross validation* pipeline. This is what you do when you're sample poor and want to test on all the data.

For example, in 20 fold cross validation we
* split the data into 20 equal folds
* train on 19 parts of the data and use the remaining fold as the test set
* repeat the experiment 20 times such that each sample has a turn being in the test set

We'd generally use heldout error for *hyperparameter selection*. We can also examine variation of performance over folds to quantify uncertainty in the holdout error.

This general setup is defined here.

In [ ]:
folds = KFold(n_splits=20)
multioutput='variance_weighted'  # desired normalisation method for "total variance in Y"

# Define return object
HoldoutResult = namedtuple('HoldoutResult', ['R2', 'R2err', 'SSE', 'fold_SSE', 'SSY', 'Y_pred'])

def holdout_err(model, X, Y, folds=folds):
    """Gets holdout mean and standard error on the R2 score."""
    
    y_pred_all = np.zeros_like(Y)
    Y_mean = Y.mean(axis=0)
    fold_SSE = []  # Fold sum squared error
    fold_SSY = []  # support non-cross splits
    for train_idx, test_idx in folds.split(X):
        model.fit(X[train_idx], Y[train_idx])
        y_pred = model.predict(X[test_idx])
        y_pred_all[test_idx] = y_pred
        fold_SSE.append(np.sum((y_pred - Y[test_idx])**2))
        fold_SSY.append(np.sum((Y[test_idx] - Y_mean)**2))

    SSE = np.sum(fold_SSE)  # sum squared error
    SSY = np.sum(fold_SSY)  # specific for non-crossval
    SE = np.std(fold_SSE, ddof=1) * np.sqrt(len(fold_SSE))  # get standard error on the SUM
    #SSY = np.sum((Y - Y.mean(axis=0))**2)  # Only true for cross-validation
    R2 = 1. - SSE / SSY
    R2err = SE / SSY
    return HoldoutResult(R2, R2err, SSE, fold_SSE, SSY, y_pred_all)


def fill_plot(fig, px, py, py_std, name, z=1., fillcolor='rgba(0,0,100,0.2)', color='rgb(0,0, 100)'):
    px = np.array(px)
    py = np.array(py)
    py_std = np.array(py_std)
    fig.add_trace(go.Scatter(x=px, y=py+z*py_std, mode='lines', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=px, y=py-z*py_std, mode='lines', line=dict(width=0),
                             fill='tonexty', fillcolor=fillcolor, showlegend=False))
    fig.add_trace(go.Scatter(x=px, y=py, name=name, line=dict(color=color)))

## Principal Component Regression with R2 score

This is a regression method that uses PCA for regularisation. You fit PCA on the features, keep only the first k principal components (those with highest variance), then regress on those components. The number of components k acts as the regularisation parameter.

We are using R2 score to quantify the fraction of variance in Y that is explained by X.

In [ ]:
# How many principle components is the best?
kvals = np.arange(1,25)
fit_r2 = []
pred_r2 = []
pred_err = []


for k in tqdm(kvals):

    # Note that in a holdout setting, we have to apply the whole methodology
    # We can't partially model (i.e. PCA transform) looking at the whole data
    # or we risk some label leakage
    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ('pca_x', PCA(n_components=k)),
            ('regressor', LinearRegression()),
        ]),
        transformer=PCA())

    # Fit error (numerically equivalent to the measure Dan proposed)
    Y_fit = model.fit(Xc, Yc).predict(Xc)
    fit_r2.append(r2_score(Yc, Y_fit, multioutput=multioutput))

    # Holdout error
    result = holdout_err(model, Xc, Yc)
    pred_r2.append(result.R2)
    pred_err.append(result.R2err)

fig = go.Figure()
fig.add_trace(go.Scatter(x=kvals, y=fit_r2, name='fit', line=dict(color='red')))
# fig.add_trace(go.Scatter(x=kvals, y=pred_r2, name='heldout'))
fill_plot(fig, kvals, pred_r2, pred_err, "heldout")
fig.update_layout(xaxis_title='Number of latent factors of X', yaxis_title='R² Y')
fig.update_layout(yaxis=dict(tick0=0, dtick=0.1))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(width=800, height=600, title="Principal Component Regression")
fig.show()


fig.write_image('plots/pcr.png')
fig.write_html("plots/pcr.html")

As the number of principal components increases, the ability to fit the target Y improves monotonically towards 100%.

However, as expected, we see a tradeoff between the number of components of X and the holdout performance of the regressor.
* Too few components and we are discarding "real" signal (underfitting)
* Too many components and the model is fitting "spurious" signals (overfitting)

The "sweet spot" at n=56 appears to be at 14 components, reconstructing ~58% of the variance in Y.
It's a lot, but no slam dunk result.

The error bars here are standard error over folds.

Note: Not every PC of X is predictive of Y - e.g. using 5 PCs of X is worse than using 3 PCs. Because our selection process is a function of X, you have to take the bad with the good. However, the value of a feature is contextual - its really a question of whether a *set* of features is predictive together, its not like we just sum up the predictiveness of each feature. Next we'll look at whether we can learned which components to use.

## Regularised Linear Regression

Given we're struggling with overfitting, I don't think the data supports a particularly complex model class.

What we can do is keep the linear model class, use all the features (or a full PCA), but use weight regularisation. Thus the options to consider are:

* L2 regularisation (Ridge regression, which favours dense weights) 
* L1 regularisation (LASSO regression, which favours sparse weights)
* L1+L2 regularisation (Elastic Net)

Let's try a 2D hyperparameter sweep and see what works.

NOTE: I've wrapped the model in a feature and a target PCA transformation.
I've tried with and without PCA, modeling with the transforms is faster and at least as good from what I've seen.
PCA transforms are re-fitted each time we train, to avoid information leaking in from the heldout samples.

In [ ]:
# What if we try learning which components to use?
# But then control overfitting using regularisation (L1 or L2)
n1 = 15
n2 = 16
L1s = np.linspace(-5., 1., n1)
L2s = np.linspace(-3., 2., n2)
r2score = np.zeros((n1, n2))

Xt = Xc
Yt = Yc

for i, j in tqdm(list(product(range(n1), range(n2)))):
    L1a = np.exp(L1s[i])
    L2a = np.exp(L2s[j])
    # model = ElasticNet(alpha=L1a + L2a, l1_ratio=L1a/(L1a + L2a), max_iter=200, warm_start=True)
    model = TransformedTargetRegressor(
        regressor=Pipeline([
            ('pca_x', PCA()),
            ('regressor', ElasticNet(alpha=L1a + L2a, l1_ratio=L1a/(L1a + L2a))),
        ]),
        transformer=PCA())
    result = holdout_err(model, Xt, Yt)
    r2score[i][j] = result.R2


plt.contourf(L1s, L2s, r2score.T, 60)
plt.xlabel("L1")
plt.ylabel("L2")
plt.title("R2 score vs regularisation")
plt.colorbar()
i, j = np.unravel_index(r2score.argmax(), r2score.shape)
plt.plot(L1s[i], L2s[j], 'ko')
plt.show()

Doesn't seem to be any advantage to mixing L1 and L2 - in fact the best result (by a small margin) is pure L1.

## Lasso (L1 regularised linear regression)

In [ ]:
# Try on the raw X without PCA first
# Such that X is using sums and differences of individual contexts
alphas = np.linspace(-2., 0., 20)
fit_r2 = []
pred_r2 = []
k_y = 4 # or None for all - but its essentially the same with 4

for a in tqdm(alphas):

    model = Lasso(alpha=np.exp(a))

    # Fit error
    Y_fit = model.fit(Xs, Yc).predict(Xs)
    model = TransformedTargetRegressor(
        Lasso(alpha=np.exp(a)),
        transformer=PCA(n_components=k_y),
        check_inverse=False,
    )

    fit_r2.append(r2_score(Yc, Y_fit, multioutput=multioutput))

    # Heldout error
    result = holdout_err(model, Xs, Yc)
    pred_r2.append(result.R2)


fig = go.Figure()
fig.add_trace(go.Scatter(x=alphas, y=fit_r2, name='fit'))
fig.add_trace(go.Scatter(x=alphas, y=pred_r2, name='heldout'))
fig.update_layout(xaxis_title='L1 regularisation', yaxis_title='R² Y')
#fig.update_layout(yaxis=dict(tick0=0, dtick=0.1))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(width=600, height=600)
fig.show()


In [ ]:
fig.write_image('plots/lasso.png')
fig.write_html("plots/lasso.html")

In [ ]:
# What are the weights behind this result?

# Fit model with best alpha on the full data
a = float(alphas[np.argmax(pred_r2)])
model = TransformedTargetRegressor(
    Lasso(alpha=np.exp(a)),
    transformer=PCA(n_components=k_y),
    check_inverse=False,  # if the PCA is lossy
)
model.fit(Xs, Yc)
coef = model.regressor_.sparse_coef_
keep = np.unique(coef.indices)
context_ids = pile.columns[keep]
z = coef.todense()
z = z[:, keep]
fig = go.Figure(data=go.Heatmap(
    z=z,
    x=context_ids,
    zmid=0,
    colorscale='RdBu',
    showscale=True
))
fig.update_layout(title='LASSO weights', 
                  xaxis_title='Context #', yaxis_title='Y PC #', yaxis=dict(autorange='reversed'),
                  width=1600, height=700)
fig.show()

fig.write_image('plots/lasso_weights.png')
fig.write_html("plots/lasso_weights.html")

In [ ]:
# What are these contexts?
import pandas as pd
texts = pd.read_csv("contexts.csv")
texts.set_index("context_id", inplace=True)
texts = texts.loc[context_ids]
texts
texts.to_csv("plots/text.csv")

# Note - why these?

There is probably nothing particularly special about any individual context, in that if we deleted a context it would just learn around that and latch onto another. Still, its interesting that it uses a bunch of enron emails, but the do seem quite diverse.


### TODO: per-component alpha
Given the PCA transformed targets are of different magnitudes (scaled by variance), its likely the optimal hyperparameters for predicting each are different. We should probably tune the hyperparameters for each Y0 inside a nested crossval loop.



In [ ]:
# Um.. that's about the same performance, but very sparse.
# 43 active features (makes sense there are less active features than data)
tol = 1.0

extract = np.where(np.abs(coef).sum(axis=0) > tol)[1]
print(f"Using {len(extract)} contexts.")
Xv = Xs[:, extract]
result = holdout_err(model, Xv, Yc)
float(result.R2)

## It feels like we're sample-limited

I going from 11 to 44 to 56 models made a big difference each time.
We can kinda see that in the follwing plot where I restrict the fraction of data we train the model on and then pick the number of latent factors based on that.

(Note that this itself is slightly optimistic, as we've used the unseen data to choose k). Noting I could also do a nested crossval, but I don't think its critical at this stage.

In [ ]:
class RandomSplit:
   def __init__(self, n_train, n_splits=1, random_state=None):
       self.k = n_train
       self.n_splits = n_splits
       self.random_state = random_state
   
   def split(self, X, y=None):
       rng = np.random.RandomState(self.random_state)
       n = len(X)
       for _ in range(self.n_splits):
           train_ix = rng.choice(n, self.k, replace=False)
           test_ix = np.setdiff1d(np.arange(n), train_ix)
           yield train_ix, test_ix

In [ ]:

n = Xc.shape[0]
#n_samples = np.linspace(10, n, 10).astype(int)  # how many samples
samples = np.arange(20, 56, 3)  # final is LOO equivalent
kvals = np.arange(1,20)
best = []
best_err = []

# TODO: quad loop: n_samples, splits, crossval, hyperpa
# Triple loop: split, hyperparameter search, split folds
for n_samples in tqdm(samples):
    my_folds = RandomSplit(n_train=n_samples, n_splits=100)


    pred_r2 = []
    pred_err = []

    for k in kvals:
        # Note that in a holdout setting, we have to apply the whole methodology
        # We can't partially model (i.e. PCA transform) looking at the whole data
        # or we risk some label leakage
        model = TransformedTargetRegressor(
            regressor=Pipeline([
                ('pca_x', PCA(n_components=k)),
                ('regressor', LinearRegression()),
            ]),
            transformer=PCA())

        # Holdout error on the train set      
        result = holdout_err(model, Xc, Yc, folds=my_folds)
        pred_r2.append(result.R2)
        pred_err.append(result.R2err)
    
    ix = np.argmax(pred_r2)  # take the best performance for any K
    best.append(pred_r2[ix])
    best_err.append(pred_err[ix])

# Plot the result
fig = go.Figure()
fill_plot(fig, samples, best, best_err, "heldout R2")
fig.update_layout(xaxis_title='n_train', yaxis_title='R² Y')
fig.update_layout(yaxis=dict(tick0=0, dtick=0.1))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(width=800, height=600, title="Reconstruction vs dataset size")
fig.show()
fig.write_image('plots/dataset_size.png')